# Objective 1b: LDA topics on the r/jobs sample

Fits LDA (latent Dirichlet allocation) to the same 10,000-comment r/jobs sample as the K-means analysis in `2_topic_discovery.ipynb`, as a comparison method. Unlike K-means, LDA gives every comment a mix of topics, which suits short comments that touch several themes.

Pipeline: Part 2's cleaned text → word counts on Part 2's vocabulary rules plus domain stop words → LDA with K = 10 → topic profiles → monthly topic shares → comparison with the K-means clusters.

Settings (chosen 2026-10-31):
- **Domain stop words.** LDA works on raw word counts, so words used everywhere on r/jobs (job, work, company, time, …) filled every topic in a first run. Part 2's `DOMAIN_STOP_WORDS` are removed, and its `PROTECTED_PHRASES` (job market, laid off, entry level, …) are kept as single tokens. The vocabulary therefore differs from the K-means one.
- **K = 10**, chosen from K = 8–20 (section 3).
- **Soft monthly shares** are the main trend measure: each month, the average topic weight over its comments. Which single topic a comment "belongs to" changes a lot between random seeds, so hard assignments are only a secondary view.

## 1. Setup

In [ ]:
# Add the repo root (and scripts/, for Part 2's word lists) to Python's search path
import sys
sys.path.append('..')
sys.path.append('../scripts')

from functions.topics import vectorize_comments, verified_coverage
from functions.topic_reporting import topic_trend_diagnostics
from functions.lda_topics import count_matrix, topic_coherence, match_topics, soft_monthly_shares
from compare_topic_models import DOMAIN_STOP_WORDS, PROTECTED_PHRASES

import os
import json
import hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

DATA_DIR = Path('../dataset/monthly_Topics')
OUT_DIR = DATA_DIR / 'lda'
FIG_DIR = OUT_DIR / 'figures'
K = 10
SEED, CHECK_SEED = 42, 7
MAX_ITER = 200
RUN_K_DIAGNOSTICS = False  # refits K = 8-20 with two seeds (~15 min); results are recorded in section 3
EARLY_MONTHS = ['2026-01', '2026-02', '2026-03']
LATE_MONTHS = ['2026-07', '2026-08', '2026-09']

# same conversational fillers as notebook 2
EXTRA_TOPIC_STOP_WORDS = [
    'just', 'like', 'really', 'actually', 'probably', 'maybe', 'yes', 'yeah',
    'lol', 'sounds', 'know', 'think', 'people', 'good', 'need', 'want',
    'going', 'got', 'did', 'does', 'said', 'say', 'don', 've', 'll', 'gif', 'giphy',
]

# named on 2026-10-31 from the top words and examples in section 5;
# 'Mixed:' marks low-coherence topics without a single theme
TOPIC_LABELS = {
    0: 'Mixed: workplace, business and services',
    1: 'Job-search difficulty, hiring process and AI',
    2: 'Offers, resigning and changing jobs',
    3: 'Pay, hours, experience and degrees',
    4: 'PTO, leave and bosses',
    5: 'Mixed: personal stories, school and tests',
    6: 'Applications, recruiters, background checks and scams',
    7: 'Mixed: life advice and skills',
    8: 'Advice, good wishes and job-market comments',
    9: 'Economy: wages, cost of living, unemployment and AI',
}

## 2. Data and word counts
Reads Part 2's files from `dataset/monthly_Topics/`. Part 2's `vectorize_comments` builds the vocabulary with its usual rules (moderation notices removed, same stop words) plus the domain stop words and protected phrases, and `count_matrix` turns that vocabulary into the raw counts LDA needs. Comments left with no words are not modelled; they stay in the export as unassigned.

In [ ]:
read = dict(dtype={'comment_id': str}, keep_default_na=False)
corpus = pd.read_csv(DATA_DIR / 'comments_clean.csv', **read)
kmeans = pd.read_csv(DATA_DIR / 'comment_topics.csv', **read)
manifest = json.loads((DATA_DIR / 'sampling_manifest.json').read_text(encoding='utf-8'))
coverage = verified_coverage(DATA_DIR / 'monthly_coverage.csv', manifest['sha256']['monthly_coverage.csv'])

features = vectorize_comments(
    corpus, min_df=3, max_df=0.90, max_features=10000,
    extra_stop_words=EXTRA_TOPIC_STOP_WORDS + DOMAIN_STOP_WORDS, keep_words=['off'],
    exclude_moderation=True, protected_phrases=PROTECTED_PHRASES,
)
docs = features['model_df']
counts, counter = count_matrix(features, docs['text_topic'], PROTECTED_PHRASES)
terms = counter.get_feature_names_out()
n_words = np.asarray(counts.sum(axis=1)).ravel()

kmeans_assigned = set(kmeans.loc[kmeans['topic_status'] == 'assigned', 'comment_id'])
print(f'{len(corpus):,} comments, {len(docs):,} modelled; all within the {len(kmeans_assigned):,} K-means comments:',
      set(docs['comment_id']) <= kmeans_assigned)
print(features['excluded'].groupby('reason').size().to_string())
print(f'Vocabulary: {len(terms):,} terms; median {np.median(n_words):.0f} counted words per comment')

## 3. Choosing K
Results of the K comparison run on 2026-10-31 (seeds 42 and 7, up to 200 iterations). Set `RUN_K_DIAGNOSTICS = True` to recompute them (about 15 minutes).

| K | NPMI mean | NPMI min | Topic diversity | Topic stability, mean / min | Comment ARI |
|---|---|---|---|---|---|
| 8 | 0.107 | 0.036 | 0.85 | 0.62 / 0.53 | 0.02 |
| **10** | **0.113** | **0.060** | 0.83 | 0.54 / 0.39 | 0.02 |
| 12 | 0.106 | 0.018 | 0.82 | 0.53 / 0.32 | 0.02 |
| 15 | 0.092 | −0.053 | 0.77 | 0.49 / 0.28 | 0.02 |
| 18 | 0.079 | −0.117 | 0.78 | 0.46 / 0.30 | 0.01 |
| 20 | 0.051 | −0.186 | 0.77 | 0.49 / 0.33 | 0.01 |

- **NPMI** (coherence): how much more often a topic's top 10 words appear together than by chance; around 0 means unrelated words.
- **Topic diversity**: share of distinct words among all topics' top 10 words; low values mean topics repeat the same words.
- **Topic stability**: cosine similarity of each topic's word distribution with its best match in the seed-7 fit.
- **Comment ARI**: agreement of each comment's top topic between the two seeds (0 = chance).

K = 10 has the highest average coherence and no weak topic. K = 8 is more stable but its topics are broader, and from K = 15 some topics have negative coherence. Comment ARI is near 0 for every K: comments are short, so single-comment assignments are not reliable and the trends below use soft shares.

In [ ]:
if RUN_K_DIAGNOSTICS:
    rows = []
    for k in [8, 10, 12, 15, 18, 20]:
        fits = [LatentDirichletAllocation(n_components=k, learning_method='batch', max_iter=MAX_ITER,
                                          evaluate_every=10, random_state=s, n_jobs=-1)
                for s in (SEED, CHECK_SEED)]
        thetas = [m.fit_transform(counts) for m in fits]
        k_coherence = topic_coherence(fits[0].components_, counts)
        k_matched = match_topics(fits[0].components_, fits[1].components_)
        top10 = [terms[w.argsort()[::-1][:10]] for w in fits[0].components_]
        rows.append({
            'k': k, 'npmi_mean': k_coherence['npmi'].mean(), 'npmi_min': k_coherence['npmi'].min(),
            'topic_diversity': len(set(np.concatenate(top10))) / (10 * k),
            'stability_mean': k_matched['cosine'].mean(), 'stability_min': k_matched['cosine'].min(),
            'comment_ari': adjusted_rand_score(thetas[0].argmax(1), thetas[1].argmax(1)),
        })
    display(pd.DataFrame(rows).round(3))

## 4. Fit LDA (K = 10)
Fits the final model with seed 42, and again with seed 7 to see which topics are stable. Each comment gets a weight for every topic (its weights sum to 1). The two fits take a few minutes.

In [ ]:
def fit_lda(seed):
    model = LatentDirichletAllocation(n_components=K, learning_method='batch', max_iter=MAX_ITER,
                                      evaluate_every=10, random_state=seed, n_jobs=-1)
    return model, model.fit_transform(counts)


lda, theta = fit_lda(SEED)
lda_check, theta_check = fit_lda(CHECK_SEED)
matched = match_topics(lda.components_, lda_check.components_)
print(f'Iterations: {lda.n_iter_} (seed {SEED}), {lda_check.n_iter_} (seed {CHECK_SEED})')
print('Comment-level ARI between seeds:', round(adjusted_rand_score(theta.argmax(1), theta_check.argmax(1)), 3))

## 5. Topic profiles
For each topic: its share (average weight over all modelled comments), top words, coherence, stability across seeds, and the five comments with the highest weight among those with at least 10 counted words (shorter ones are often a single word). The names in `TOPIC_LABELS` (section 1) were chosen from these. They only fit this model: if the data, settings or seed change, rename the topics, since topic numbers are not carried over.

In [ ]:
coherence = topic_coherence(lda.components_, counts)
order = np.argsort(lda.components_, axis=1)[:, ::-1]
profiles = pd.DataFrame({
    'share': theta.mean(axis=0),
    'top_words': [', '.join(terms[row[:12]]) for row in order],
    'npmi': coherence['npmi'],
    'stability': matched.set_index('topic')['cosine'],
}).rename_axis('topic')
profiles.insert(0, 'label', [TOPIC_LABELS.get(t, ', '.join(terms[order[t, :3]])) for t in range(K)])
chart_labels = {t: f'T{t} {label}' for t, label in profiles['label'].items()}

pd.set_option('display.max_colwidth', 120)
profiles.round(3)

In [ ]:
# the five highest-weight comments per topic, among comments with at least 10 counted words
long_enough = np.flatnonzero(n_words >= 10)
examples = []
for t in range(K):
    for i in long_enough[np.argsort(theta[long_enough, t])[::-1][:5]]:
        examples.append({'topic': t, 'weight': round(theta[i, t], 2),
                         'comment_id': docs['comment_id'].iloc[i], 'text': docs['text'].iloc[i][:220]})
examples = pd.DataFrame(examples)
pd.set_option('display.max_colwidth', 220)
examples

## 6. Monthly topic shares
**Soft shares** (main measure): each month, the average topic weight over that month's comments. The seed-7 fit, with its topics matched to seed 42, shows how much the trends depend on the random seed. January–March is compared with July–September; October is partial (sample cut off on 28 October) and marked with `*`. Every month has the same 1,000-comment quota, so shares show relative attention, not discussion volume.

In [ ]:
shares = soft_monthly_shares(theta, docs['month'])
shares_check = soft_monthly_shares(theta_check, docs['month'])[matched['matched_topic'].to_numpy()]
shares_check.columns = matched['topic'].to_numpy()

partial = set(coverage.loc[coverage['partial_month'].astype(str).str.lower() == 'true', 'month'])
month_ticks = [m[5:] + ('*' if m in partial else '') for m in shares.index]


def period_change(monthly):
    early, late = monthly.loc[EARLY_MONTHS].mean(), monthly.loc[LATE_MONTHS].mean()
    return pd.DataFrame({'jan_mar': early, 'jul_sep': late, 'change_pp': (late - early) * 100})


trend = period_change(shares).join(period_change(shares_check)['change_pp'].rename('change_pp_seed7'))
trend['same_direction'] = np.sign(trend['change_pp']) == np.sign(trend['change_pp_seed7'])
trend.insert(0, 'label', profiles['label'])
trend.round(3)

In [ ]:
os.makedirs(FIG_DIR, exist_ok=True)

fig, ax = plt.subplots(figsize=(13, 6))
image = ax.imshow(shares.T.to_numpy() * 100, aspect='auto', cmap='Blues')
ax.set_xticks(range(len(month_ticks)), month_ticks)
ax.set_yticks(range(K), [chart_labels[t] for t in range(K)])
for row in range(K):
    for col in range(len(shares)):
        ax.text(col, row, f'{shares.iloc[col, row] * 100:.1f}', ha='center', va='center', fontsize=8)
ax.set(xlabel='2026 UTC month (* partial)', title='LDA soft topic shares per month (%)')
fig.colorbar(image, ax=ax)
plt.tight_layout()
plt.savefig(FIG_DIR / 'soft_shares_heatmap.png', dpi=150)
plt.show()

fig, axes = plt.subplots(2, 5, figsize=(20, 7), sharex=True, sharey=True)
x = range(len(shares))
for t, ax in zip(range(K), axes.ravel()):
    ax.plot(x, shares[t] * 100, marker='o', label=f'seed {SEED}')
    ax.plot(x, shares_check[t] * 100, linestyle='--', label=f'seed {CHECK_SEED} (matched)')
    ax.set_title(chart_labels[t], fontsize=9)
    ax.set_xticks(list(x), month_ticks, fontsize=7)
for ax in axes[:, 0]:
    ax.set_ylabel('Soft share (%)')
axes[0, 0].legend(fontsize=7)
plt.tight_layout()
plt.savefig(FIG_DIR / 'soft_shares_by_topic.png', dpi=150)
plt.show()

### Secondary: top topic per comment
Each comment assigned to its highest-weight topic and passed to Part 2's `topic_trend_diagnostics`, so the output has the same form as the K-means trends (including equal weight per post). Given the low comment-level stability, use this only as a cross-check of the soft shares.

In [ ]:
hard = docs.assign(cluster=theta.argmax(axis=1))
_, _, hard_monthly, hard_summary = topic_trend_diagnostics(
    hard, coverage[['month', 'partial_month', 'source_window_complete']], EARLY_MONTHS, LATE_MONTHS,
)
hard_summary.insert(1, 'label', hard_summary['cluster'].map(profiles['label']))
hard_summary[['cluster', 'label', 'n_comments', 'first3_share', 'last3_share', 'change_pp',
              'equal_post_change_pp']].round(3)

## 7. Comparison with K-means
For each K-means cluster (with Part 2's suggested names), the average LDA topic mix of its comments: where its content goes in the LDA model. Rows sum to 100%. NMI compares the K-means clusters with each comment's LDA top topic (0 = unrelated, 1 = the same partition); it inherits the instability of the top topic, so read the broad pattern rather than single cells.

In [ ]:
km = kmeans.set_index('comment_id').loc[docs['comment_id']]
km_cluster = km['cluster'].astype(float).astype(int).to_numpy()
km_names = km.assign(cluster=km_cluster).groupby('cluster')['topic_label'].first()

mix = pd.DataFrame(theta).groupby(km_cluster).mean()
mix.index = [f'C{c} {km_names[c]}' for c in mix.index]
mix.columns = [chart_labels[t] for t in mix.columns]
print('NMI, K-means clusters vs LDA top topics:',
      round(normalized_mutual_info_score(km_cluster, theta.argmax(axis=1)), 3))

fig, ax = plt.subplots(figsize=(14, 9))
image = ax.imshow(mix.to_numpy() * 100, aspect='auto', cmap='Purples')
ax.set_xticks(range(K), mix.columns, rotation=40, ha='right', fontsize=8)
ax.set_yticks(range(len(mix)), mix.index, fontsize=8)
for row in range(len(mix)):
    for col in range(K):
        ax.text(col, row, f'{mix.iloc[row, col] * 100:.0f}', ha='center', va='center', fontsize=7)
ax.set(xlabel='LDA topic', ylabel='K-means cluster', title='Average LDA topic mix of each K-means cluster (%)')
fig.colorbar(image, ax=ax)
plt.tight_layout()
plt.savefig(FIG_DIR / 'kmeans_vs_lda.png', dpi=150)
plt.show()

## 8. Export
Writes to `dataset/monthly_Topics/lda/` (ignored by git):
- `lda_comment_topics.csv`: all 10,000 comments with their status, top topic, its weight and the weight of every topic.
- `lda_topic_profiles.csv`, `lda_monthly_shares.csv` (seed 42 and matched seed 7), `lda_trend_summary.csv`, `kmeans_lda_mix.csv`.
- `lda_run.json` with the settings and input hash, and the figures above in `figures/`.

In [ ]:
weights = pd.DataFrame(theta, columns=[f'topic_{t}' for t in range(K)])
modelled = docs[['comment_id', 'month']].reset_index(drop=True).join(weights)
modelled.insert(2, 'lda_status', 'modelled')
modelled.insert(3, 'top_topic', theta.argmax(axis=1))
modelled.insert(4, 'top_weight', theta.max(axis=1))
unmodelled = features['excluded'].rename(columns={'reason': 'lda_status'})
unmodelled['lda_status'] = unmodelled['lda_status'].replace({'zero_tfidf': 'no_words_left'})
comment_topics = pd.concat([modelled, unmodelled], ignore_index=True)
comment_topics['top_topic'] = comment_topics['top_topic'].astype('Int64')
assert len(comment_topics) == len(corpus) and comment_topics['comment_id'].is_unique

monthly = pd.concat({'seed_42': shares.stack(), 'seed_7_matched': shares_check.stack()}, axis=1)
monthly = monthly.rename_axis(['month', 'topic']).reset_index()

comment_topics.to_csv(OUT_DIR / 'lda_comment_topics.csv', index=False)
profiles.to_csv(OUT_DIR / 'lda_topic_profiles.csv')
monthly.to_csv(OUT_DIR / 'lda_monthly_shares.csv', index=False)
trend.to_csv(OUT_DIR / 'lda_trend_summary.csv')
mix.to_csv(OUT_DIR / 'kmeans_lda_mix.csv')
run = {
    'k': K, 'seed': SEED, 'check_seed': CHECK_SEED, 'max_iter': MAX_ITER,
    'iterations': [int(lda.n_iter_), int(lda_check.n_iter_)],
    'modelled_comments': len(docs), 'vocabulary_size': len(terms),
    'stop_words': 'sklearn English + notebook 2 fillers + compare_topic_models.DOMAIN_STOP_WORDS; keep "off"',
    'protected_phrases': PROTECTED_PHRASES, 'topic_labels': TOPIC_LABELS,
    'input_sha256': hashlib.sha256((DATA_DIR / 'comments_clean.csv').read_bytes()).hexdigest(),
}
(OUT_DIR / 'lda_run.json').write_text(json.dumps(run, indent=2), encoding='utf-8')
print('Saved to', OUT_DIR)

## 9. Notes
- LDA is a comparison with the proposal's K-means, not a replacement. Its vocabulary differs (domain stop words), and the comments left with no words are not modelled.
- Comments are short, so which topic a single comment belongs to is unstable between seeds. Read trends from the soft shares and check the seed-7 column before describing a change.
- Every month has the same 1,000-comment quota, so shares show relative attention, not volume. October is partial.
- Topic names are the group's reading of the top words and example comments, not ground truth.